# 3. Inputs, sensitivity and optimisation

From data to a decision: fit an input distribution, find which parameters matter, approximate the
response with a surrogate, and optimise a cost.

In [ ]:
import numpy as np

from simulsi.randomness import fit_distribution

observed = np.random.default_rng(3).gamma(2.0, 2.5, size=300)  # stand-in for measured service times
fits = fit_distribution(observed)
fits

In [ ]:
service = fits.best.distribution
print(service, service.mean)

## Which inputs matter?

Morris screening costs `r * (d + 1)` runs and ranks inputs by `mu*` (mean absolute effect).

In [ ]:
from simulsi.analysis import morris_screening
from simulsi.models import mmc

model = mmc.with_options(duration=1_000, warmup=100)
screen = morris_screening(
    model,
    {"arrival_rate": (1.0, 1.8), "service_rate": (0.9, 1.1), "servers": (2, 4)},
    r=6,
    outputs=["resource.server.wait.mean"],
    seed=1,
)
screen

## A surrogate of the response surface

In [ ]:
from simulsi.optimization import fit_surrogate

surface = fit_surrogate(
    model,
    {"arrival_rate": (1.0, 1.8), "servers": (2, 4)},
    ["resource.server.wait.mean"],
    n=20,
    seed=2,
)
print(surface.accuracy())
mean, sd = surface.predict({"arrival_rate": 1.5, "servers": 3}, return_std=True)
print(f"predicted wait {mean[0]:.3f} +/- {sd[0]:.3f}")

## Optimise staffing against a cost

Each server costs 20 per time unit; each waiting customer costs 50 per time unit.
`optimize` searches, then re-checks the best candidates with fresh seeds.

In [ ]:
from simulsi.optimization import optimize


def cost(m):
    servers = m["resource.server.capacity_time"] / m["sim.observed_time"]
    return 20 * servers + 50 * m["resource.server.mean_queue_length"]


best = optimize(
    model,
    None,
    {"servers": (1, 6)},
    fixed={"arrival_rate": 1.6},
    method="grid",
    budget=6,
    replications=3,
    transform=cost,
)
best